In [8]:
import numpy as np
from sklearn.preprocessing import StandardScaler
import eval
import evaluation
from align import extract_word_embeddings
from TextCore import TextCore

data = np.load(r'data\restaurant_review.npy', allow_pickle=True)
# data = np.load(r'data\sms_spam.npy', allow_pickle=True)
# data = np.load(r'data\grammar.npy', allow_pickle=True)

data = data.item()

print(type(data))        
print(data.keys())       
tokens = data['tokens']
labels = data['labels']
sentence_labels = [1 if any(labels) else 0 for labels in labels]

X_train, y_train, X_test, y_test, z_train, z_test = eval.split_anomaly_data(tokens, sentence_labels, labels)

<class 'dict'>
dict_keys(['sentences', 'tokens', 'labels'])


In [9]:
model_name = 'bert-base-cased'
local_cache_dir = r'D:\models\bert'

In [10]:
train_data = extract_word_embeddings(
        model=None,
        tokenizer=None,
        all_tokens=X_train,
        all_labels=z_train,
        model_name=model_name,
        local_cache_dir = local_cache_dir,
        device="cuda"
    )
X_train = np.array([item['embedding'] for sent in train_data for item in sent])
z_train = [item['label'] for sent in train_data for item in sent]

Using device: cuda

Sentence 1/550: Great place for great food . I ' ll be back
Tokenization alignment:
----------------------------------------
  [Special] [CLS]      -> position 0 (ignored)
  Token  0 'Great     ' -> subword 'Great     ' at position 1 (used)
  Token  1 'place     ' -> subword 'place     ' at position 2 (used)
  Token  2 'for       ' -> subword 'for       ' at position 3 (used)
  Token  3 'great     ' -> subword 'great     ' at position 4 (used)
  Token  4 'food      ' -> subword 'food      ' at position 5 (used)
  Token  5 '.         ' -> subword '.         ' at position 6 (used)
  Token  6 'I         ' -> subword 'I         ' at position 7 (used)
  Token  7 ''         ' -> subword ''         ' at position 8 (used)
  Token  8 'll        ' -> subword 'll        ' at position 9 (used)
  Token  9 'be        ' -> subword 'be        ' at position 10 (used)
  Token 10 'back      ' -> subword 'back      ' at position 11 (used)
  [Special] [SEP]      -> position 12 (ignored)

In [11]:
test_data = extract_word_embeddings(
        model=None,
        tokenizer=None,
        all_tokens=X_test,
        all_labels=z_test,
        model_name=model_name,
        local_cache_dir = local_cache_dir,
        device="cuda"
    )
X_test = np.array([item['embedding'] for sent in test_data for item in sent])
z_test = [item['label'] for sent in test_data for item in sent]

Using device: cuda

Sentence 1/550: Nice chill brunch spot . Amazing focaccia pizza ! Light and just right . Potato breakfast was yum ! Recommend for jammy eggs . Atmosphere is chill and the lighting is so romantic .
Tokenization alignment:
----------------------------------------
  [Special] [CLS]      -> position 0 (ignored)
  Token  0 'Nice      ' -> subword 'Nice      ' at position 1 (used)
  Token  1 'chill     ' -> subword 'chill     ' at position 2 (used)
  Token  2 'brunch    ' -> subword 'br        ' at position 3 (used)
           subword '##unch    ' at position 4 (skipped)
  Token  3 'spot      ' -> subword 'spot      ' at position 5 (used)
  Token  4 '.         ' -> subword '.         ' at position 6 (used)
  Token  5 'Amazing   ' -> subword 'Amazing   ' at position 7 (used)
  Token  6 'focaccia  ' -> subword 'f         ' at position 8 (used)
           subword '##oc      ' at position 9 (skipped)
           subword '##ac      ' at position 10 (skipped)
           subword 

In [12]:
model = TextCore()
model.fit(X_train)
score = model.decision_function(X_test)

from sklearn.metrics import roc_auc_score, precision_recall_curve, auc
roc_auc = roc_auc_score(z_test, score)
roc_auc

0.8291871975956033